# V-JEPA 2 perception — Colab smoke test

Goal: validate the precompute pipeline end-to-end on a tiny slice of LIBERO Object using a Colab T4.

Budget: 2 episodes, frame stride 4, ViT-L float16 — should take a few minutes on T4.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install deps

If your branch is private, swap the URL for a token-authenticated one or upload the folder manually.

In [ ]:
%cd /content
!git clone -b feat/vjepa2-perception https://github.com/PoCInnovation/LeWM-Robot.git || (cd LeWM-Robot && git fetch && git checkout feat/vjepa2-perception && git pull)
%cd /content/LeWM-Robot/vjepa-perception

In [ ]:
!pip install -q -r requirements.txt

## 3. (Optional) Hugging Face login

Run this only if model or dataset access requires it. `facebook/vjepa2-vitl-fpc64-256` and `lerobot/libero_object_image` are public at time of writing — feel free to skip.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4. Run the precompute (smoke test)

- `--max_episodes 2` — only 2 episodes
- `--frame_stride 4` — keep 1 frame out of 4
- `--batch_size 8` — gentle on T4 VRAM

In [ ]:
!python precompute_features.py \
    --src_repo lerobot/libero_object_image \
    --dst_dir /content/cached_features/libero_object_smoke \
    --vjepa2_repo facebook/vjepa2-vitl-fpc64-256 \
    --batch_size 8 \
    --dtype float16 \
    --device cuda \
    --num_workers 2 \
    --max_episodes 2 \
    --frame_stride 4

## 5. Inspect the cache

Sanity-check shapes, dtype, and that the sliding-window loader can produce a batch.

In [ ]:
import json, pathlib
cache = pathlib.Path('/content/cached_features/libero_object_smoke')
meta = json.loads((cache / 'metadata.json').read_text())
print(json.dumps(meta, indent=2)[:1200])

In [ ]:
from cached_loader import CachedVJepa2Dataset
from torch.utils.data import DataLoader

ds = CachedVJepa2Dataset(cache, clip_length=1, stride=1)
print(f'clips: {len(ds)}')
sample = ds[0]
for k, v in sample.items():
    print(f'  {k}: {tuple(v.shape) if hasattr(v, "shape") else v}  {getattr(v, "dtype", "")}')

In [ ]:
dl = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
batch = next(iter(dl))
for k, v in batch.items():
    print(f'  {k}: {tuple(v.shape) if hasattr(v, "shape") else v}')

## 6. Next steps

If everything above ran cleanly:
1. Re-run cell 4 with `--max_episodes 5` (and optionally `--frame_stride 2`) to build the real Phase 1 cache.
2. Move to Phase 2: implement `policy_head.py` (mean-pool + small MLP).